In [1]:
import pandas as pd
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from nltk.corpus import stopwords

In [2]:
import pandas as pd
import requests, zipfile, io

In [3]:
url = "http://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip"
response = requests.get(url)

In [4]:
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open("training.1600000.processed.noemoticon.csv") as f:
        twitter_data = pd.read_csv(
            f,
            names=['target','id','date','flag','user','text'],
            encoding='latin-1'
        )

In [5]:
print(twitter_data.shape)
print(twitter_data.head())

(1600000, 6)
   target          id                          date      flag  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                               text  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  


In [6]:
twitter_data.shape

(1600000, 6)

In [7]:
twitter_data.head()

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [8]:
twitter_data.isnull().sum()

,0
target,0
id,0
date,0
flag,0
user,0
text,0


In [9]:
twitter_data['target'].value_counts()

,count
target,
0,800000
4,800000


In [10]:
twitter_data.replace({'target':{4:2}}, inplace=True)

In [11]:
twitter_data['target'].value_counts()

,count
target,
0,800000
2,800000


In [12]:
import re
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk


In [13]:
# Download stopwords if not already done
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [14]:
port_stem = PorterStemmer()
def stemming(content):
  stemmed_content = re.sub('[^a-zA-Z]',' ',content)
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ' '.join(stemmed_content)

  return stemmed_content
stop_words = set(stopwords.words('english'))
porter_stemmer = PorterStemmer()

def stemming(content):
    content = re.sub('[^a-zA-Z]', ' ', content).lower()
    words = content.split()
    stemmed_words = [porter_stemmer.stem(word) for word in words if word not in stop_words]
    stemmed_content = ' '.join(stemmed_words)
    return stemmed_content

In [15]:
twitter_data['stemmed_content'] = twitter_data['text'].apply(stemming)

In [16]:
twitter_data.head()

,target,id,date,flag,user,text,stemmed_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see


In [17]:
print(twitter_data['stemmed_content'])

0          switchfoot http twitpic com zl awww bummer sho...
1          upset updat facebook text might cri result sch...
2          kenichan dive mani time ball manag save rest g...
3                            whole bodi feel itchi like fire
4                              nationwideclass behav mad see
                                 ...                        
1599995                           woke school best feel ever
1599996    thewdb com cool hear old walt interview http b...
1599997                         readi mojo makeov ask detail
1599998    happi th birthday boo alll time tupac amaru sh...
1599999    happi charitytuesday thenspcc sparkschar speak...
Name: stemmed_content, Length: 1600000, dtype: object


In [18]:
print(twitter_data['target'])

0          0
1          0
2          0
3          0
4          0
          ..
1599995    2
1599996    2
1599997    2
1599998    2
1599999    2
Name: target, Length: 1600000, dtype: int64


In [19]:
#seperation
X = twitter_data['stemmed_content'].values
Y = twitter_data['target'].values

In [20]:
print(X)

['switchfoot http twitpic com zl awww bummer shoulda got david carr third day'
 'upset updat facebook text might cri result school today also blah'
 'kenichan dive mani time ball manag save rest go bound' ...
 'readi mojo makeov ask detail'
 'happi th birthday boo alll time tupac amaru shakur'
 'happi charitytuesday thenspcc sparkschar speakinguph h']


In [21]:
print(Y)

[0 0 0 ... 2 2 2]


In [22]:
#Spitting data to training and test data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.9, stratify=Y, random_state=42)

In [23]:
print(X.shape, X_train.shape, X_test.shape)

(1600000,) (160000,) (1440000,)


In [24]:
print(X_train)

['forev sickest kid soooooooo ace thank lord given someon jesu obv talk'
 'bike race prepar begin' 'bore mind shoolay' ... 'drown nake reveng'
 'fix practic' 'strwbrri sweet right bore']


In [25]:
print(X_test)

['poprock http twitpic com wxx hehe love'
 'bmk yup got tru today play young kid amp initi enthusiast worth'
 'garydzen hey gari guess im boston appl store boyleston st use imac lol b b day'
 ...
 'sorri updat twitter much use ive busi crappi summer program like said'
 'p venceram ok menina eu fiz um twitter'
 'brianjshoopman thank boat tri finish new collect probabl start new book']


In [26]:
#feature_extraction(text data to numeric data)

vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [27]:
print(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1182702 stored elements and shape (160000, 110017)>
  Coords	Values
  (0, 32759)	0.2666161144061421
  (0, 87226)	0.3826497004127957
  (0, 51463)	0.2221568118793952
  (0, 89541)	0.33987450445649736
  (0, 609)	0.3209273685601872
  (0, 95545)	0.1556047217547542
  (0, 57128)	0.30237078903374537
  (0, 35742)	0.2994444735700369
  (0, 89356)	0.21047519758568822
  (0, 46987)	0.31293270442922233
  (0, 70388)	0.35440128480510663
  (0, 94009)	0.203709296207864
  (1, 9902)	0.4888839227689517
  (1, 78295)	0.5041352535473768
  (1, 76507)	0.5090296167638134
  (1, 8818)	0.4977238244710102
  (2, 11574)	0.37593385850209077
  (2, 63128)	0.44188960997014093
  (2, 86821)	0.8144981931423585
  (3, 36322)	0.16042525530047166
  (3, 89636)	0.2916148425884128
  (3, 97087)	0.2988573627407575
  (3, 39058)	0.2335425182609089
  (3, 627)	0.3112244868734948
  (3, 92052)	0.4167954364831894
  :	:
  (159995, 78973)	0.464092881425974
  (159995, 14315)	0.5586101

In [28]:
print(X_test)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9924788 stored elements and shape (1440000, 110017)>
  Coords	Values
  (0, 18873)	0.26617322542943833
  (0, 39359)	0.4048133369288649
  (0, 41403)	0.2476377147741085
  (0, 57386)	0.2408956925025383
  (0, 75992)	0.7367795815921643
  (0, 100451)	0.32105864263878936
  (1, 3483)	0.18607738920482222
  (1, 28907)	0.43654164054998135
  (1, 36572)	0.17355002051049567
  (1, 43522)	0.390604380356134
  (1, 51463)	0.2509693339796077
  (1, 75360)	0.2191282350641537
  (1, 98064)	0.1685403997979867
  (1, 99565)	0.41325431038147953
  (1, 106633)	0.2915020348948641
  (1, 108899)	0.3203130709472611
  (1, 109208)	0.3071701947440656
  (2, 4907)	0.3109891688433613
  (2, 11672)	0.3577778857242756
  (2, 22600)	0.15733717647110776
  (2, 34486)	0.41953610153995025
  (2, 37499)	0.24867887153443674
  (2, 39831)	0.22817345000653114
  (2, 42895)	0.18889090542559872
  (2, 42897)	0.4118772530492376
  :	:
  (1439997, 20171)	0.3798363635216827
  (1439997, 4

In [29]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = DecisionTreeClassifier()
decision_tree_model.fit(X_train, Y_train)

DecisionTreeClassifier()

In [30]:
decision_tree_train_prediction = decision_tree_model.predict(X_train)
decision_tree_training_data_accuracy = accuracy_score(Y_train, decision_tree_train_prediction)
print('Accuracy score on training data (Decision Tree):', decision_tree_training_data_accuracy * 100)

decision_tree_test_prediction = decision_tree_model.predict(X_test)
decision_tree_test_data_accuracy = accuracy_score(Y_test, decision_tree_test_prediction)
print('Accuracy score on test data (Decision Tree):', decision_tree_test_data_accuracy * 100)

Accuracy score on training data (Decision Tree): 99.811875
Accuracy score on test data (Decision Tree): 69.41770833333332


In [31]:
# Overall accuracy
overall_accuracy = (decision_tree_training_data_accuracy + decision_tree_test_data_accuracy) / 2
print('Overall accuracy:', overall_accuracy * 100)

Overall accuracy: 84.61479166666666


In [33]:
import pickle

In [34]:
filename = 'trained_model.sav'
pickle.dump(decision_tree_model, open(filename, 'wb'))

In [35]:
# Save the vectorizer
filename_vectorizer = 'vectorizer.sav'
pickle.dump(vectorizer, open(filename_vectorizer, 'wb'))

In [36]:
import pickle

loaded_model = pickle.load(open("trained_model.sav", "rb"))

In [40]:
X_new = X_test[200]
print(Y_test[200])
prediction = loaded_model.predict(X_new)
print(prediction)

if (prediction[0] == 0):
  print('Negative Tweet')
else:
  print('Positive Tweet')

2
[2]
Positive Tweet


In [41]:
X_new = X_test[3]
print(Y_test[3])
prediction = loaded_model.predict(X_new)
print(prediction)

if (prediction[0] == 0):
  print('Negative Tweet')
else:
  print('Positive Tweet')

2
[2]
Positive Tweet


In [42]:
# Importing necessary libraries
import re
import pickle
from nltk.corpus import stopwords

# Function for text preprocessing
def preprocess_text(text):
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower()
    text = text.split()
    text = [word for word in text if not word in set(stopwords.words('english'))]
    text = ' '.join(text)
    return text

In [43]:
# Load the pre-trained model
loaded_model = pickle.load(open('trained_model.sav', 'rb'))

In [44]:
# Function for sentiment analysis
def predict_sentiment(input_text):
    # Preprocess the input text
    input_text = preprocess_text(input_text)
    # Vectorize the input text using the same vectorizer as during training
    input_vectorized = vectorizer.transform([input_text])
    # Make prediction
    prediction = loaded_model.predict(input_vectorized)
    # Display the result
    if prediction[0] == 0:
        return 'Negative Tweet'
    else:
        return 'Positive Tweet'

In [45]:
vectorizer = pickle.load(open("vectorizer.sav", "rb"))

In [46]:
# Example usage
user_input = input("Enter a tweet: ")
result = predict_sentiment(user_input)
print("User Input : ",user_input)
print("Prediction:", result)

Enter a tweet: he was soo rude in the video
User Input :  he was soo rude in the video
Prediction: Negative Tweet
